# DCF from statement UFCF

Project **unlevered free cash flow** in the model, then call `evaluate_dcf` with **WACC** and a **terminal value** specification. Bridge **enterprise value to equity** with net debt or an equity bridge.

## Concept

`evaluate_dcf` discounts **forecast-period** UFCF, adds **PV of terminal value** (Gordon growth or exit multiple), and subtracts **net debt** (from `total_debt` / `cash` at the valuation boundary) unless you pass `net_debt_override`. Set the ISO currency directly with `ModelBuilder.with_meta` before `build()`.

In [ ]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import series
from finstack_quant.core.money import Money
from finstack_quant.statements import ModelBuilder
from finstack_quant.statements_analytics import evaluate_dcf, wacc as wacc_of

PERIODS = ["2026", "2027", "2028", "2029", "2030"]

b = ModelBuilder("dcf-demo")
b.periods("2026..2030", "2025")  # all forecast for UFCF stream
b.with_meta("currency", '"USD"')
b.value_money("ufcf", [(period, Money(value, "USD")) for period, value in zip(PERIODS, [18.0, 19.5, 21.0, 22.0, 23.0], strict=True)])
b.value("total_debt", series(PERIODS, [80.0, 78.0, 75.0, 72.0, 70.0]))
b.value("cash", series(PERIODS, [12.0, 13.0, 14.0, 15.0, 16.0]))
spec = b.build()
model_json = spec.to_json()

# Illustrative WACC build. The blend is Rust-owned: `statements_analytics.wacc`
# applies w_E * r_E + w_D * r_D * (1 - T) and validates the weights and tax rate.
cost_equity = 0.12
cost_debt_pretax = 0.06
tax_rate = 0.25
debt_weight = 0.35
equity_weight = 0.65
wacc = wacc_of(equity_weight, cost_equity, debt_weight, cost_debt_pretax, tax_rate)
print("Illustrative WACC build:", round(wacc, 4), "(used in evaluate_dcf below)")

tv_gordon = json.dumps({"type": "gordon_growth", "growth_rate": 0.025})
tv_exit = json.dumps({"type": "exit_multiple", "terminal_metric": 24.0, "multiple": 9.0})

out_g = evaluate_dcf(model_json, wacc, tv_gordon, ufcf_node="ufcf")
# Monetary fields are `Money` wire objects: {"amount": "<decimal string>", "currency": "USD"}.
def amt(money):
    return float(money["amount"])

print("Gordon terminal — equity_value:", amt(out_g["equity_value"]), "EV:", amt(out_g["enterprise_value"]))

out_x = evaluate_dcf(model_json, wacc, tv_exit, ufcf_node="ufcf")
print("Exit multiple terminal — equity_value:", amt(out_x["equity_value"]))

bridge = json.dumps({
    "total_debt": 70.0,
    "cash": 16.0,
    "preferred_equity": 0.0,
    "minority_interest": 0.0,
    "non_operating_assets": 2.0,
    "other_adjustments": [["pension", 1.0]],
})
out_br = evaluate_dcf(
    model_json,
    wacc,
    tv_gordon,
    ufcf_node="ufcf",
    equity_bridge_json=bridge,
)
print("With equity_bridge_json — equity_value:", amt(out_br["equity_value"]), "net_debt field:", amt(out_br["net_debt"]))


In [ ]:
print("DCF sensitivity grid")
# Sensitivity grid: WACC vs terminal growth (Gordon)

growth_rates = [0.015, 0.025, 0.035]
waccs = [0.085, 0.095, 0.105]
print("WACC / growth -> equity_value (Gordon)")
for w in waccs:
    row = []
    for g in growth_rates:
        if g >= w:
            row.append(None)
            continue
        tv = json.dumps({"type": "gordon_growth", "growth_rate": g})
        o = evaluate_dcf(model_json, w, tv, ufcf_node="ufcf")
        row.append(round(amt(o["equity_value"]), 2))
    print(w, row)
print("Done.")

## Takeaways

- DCF needs **forecast UFCF** rows, **`meta.currency`**, and **balance-sheet anchors** (or overrides) for net debt.
- **Terminal value** tagging follows `TerminalValueSpec` JSON: `gordon_growth` vs `exit_multiple`.
- **`statements_analytics.wacc`** owns the capital-structure blend; `evaluate_dcf` still takes WACC as a **scalar assumption** by design, so the two compose without coupling.
- A full **WACC** build belongs in your assumptions; the grid shows how sensitive equity value is to **rate + terminal growth**.